# BERT와 ELECTRA 모델 비교 실습

- 이번 복습과제에서는 SST-2 데이터셋을 기반으로 BERT와 ELECTRA 모델을 학습시켜보고 성능과 구조의 차이를 알아보겠습니다.
- 코드 실행시간이 매우 길 수 있습니다.
  - 최대한 끝까지 실행해보시되, 시간 부족으로 인해 중간에 중지하신 실행 결과를 제출하셔도 괜찮습니다.
  - 제출 이후에는 꼭 끝까지 실행시켜 비교해보시기 바랍니다!

In [1]:
!pip install --upgrade --quiet datasets fsspec huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.0/201.0 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.0/521.0 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 15.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.3 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 1.2.3 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.10.0 which is incompatible.


---------------
여기까지만 실행
---------------
그 다음,  런타임 > 세션 다시 시작 > 아래 셀부터 실행

In [ ]:
!pip install -U transformers huggingface-hub==0.34. datasets

In [2]:
!pip install huggingface-hub==0.34.0 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.7/558.7 kB 20.6 MB/s eta 0:00:00


In [2]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from torch.optim import AdamW
from tqdm import tqdm

In [3]:
# batch_size와 epochs를 조정해보세요!
batch_size = 16
epochs = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
# 데이터셋 로드
raw_datasets = load_dataset("sst2")
raw_datasets

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

In [5]:
# 전처리
def tokenize_function(examples, tokenizer):
    return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=128)

## 🔹 BERT와 ELECTRA 실험

In [ ]:
# 학습 함수 정의
def train_and_evaluate(model_name):
    print(f"\n======== Now Training: {model_name} ========")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenized_datasets = raw_datasets.map(lambda x: tokenize_function(x, tokenizer), batched=True)

    tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
    tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    train_dataset = tokenized_datasets["train"]
    valid_dataset = tokenized_datasets["validation"]

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    model.train()
    for epoch in range(2):
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            loss = outputs.loss
            total_loss += loss.item()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        avg_loss = total_loss/ len(train_loader)
        print(f"Epoch {epoch+1} - Avg Train Loss: {avg_loss:.4f}")

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in valid_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            predictions = torch.argmax(outputs.logits, dim=-1)
            correct += (predictions == batch["labels"]).sum().item()
            total += batch["labels"].size(0)

    acc = correct / total
    print(f"Validation Accuracy ({model_name}): {acc:.4f}")
    return acc

# 실행 및 평가
bert_acc = train_and_evaluate("bert-base-uncased")
electra_acc = train_and_evaluate("google/electra-base-discriminator")


======== Now Training: bert-base-uncased ========


Map:   0%|          | 0/872 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1: 100%|██████████| 4210/4210 [23:01<00:00,  3.05it/s]


Epoch 1 - Avg Train Loss: 0.1978


Epoch 2:  50%|████▉     | 2096/4210 [11:29<11:34,  3.04it/s]

## 📊 결과 비교 및 분석

아래 항목에 대한 답을 간략히 적어주세요:

1. 각 모델 구조 설명
2. 어떤 모델이 적합한지에 대한 본인의 의견
  - 학습 속도, accuracy 등 고려


# 1.
## BERT
- 트랜스포머 기반 언어 모델로 양방향 인코더를 사용하여 문장을 양방향으로 이해할 수 있다. 입력 문장에서 임의로 단어를 마스킹하고 예측하는 방식으로 사전 학습을 진행하고 다양한 토큰을 사용하여 문장 작업에 이용한다.
## ELECTRA
- 트랜스포머 기반 언어 모델로, 입력 마스킹 대신 언어 모델링 방식을 사용하거나 일부 토큰을 MASK로 대체 후 복원하는 방식이다. 사전 학습에서 트랜스포머 인코더 구조의 생성 모델과 판별을 사용한다. 다운스트림 작업 시에는 판별 모델만 ㅏ용하며 BERT와 동일한 구조를 갖는다.
# 2.
- ELECTRA는 GAN을 사용해 학습하므로 BERT에 비해 더 효율적인 학습이 가능해 대규모 데이터셋에서 모델을 더 빠르게 학습할 수 있다. GAN을 사용해 토큰을 생성하기 때문에 더 자연스러운 문장을 생성하고 모델의 매개변수가 더 적다는 특징이 있어 ELECTRA가 BERT보다 더 적합할 것이라고 생각한다.